RF + AE + BGWO for feature selection 

IMPORTANT NOTE:

**USING CV = 5**
**CLASS_WEIGHT = 'BALANCED'**

FOR EXAMPLE OUTPUTS WARNINGS BECAUSE A CLASS HAS 2 MEMBERS ONLY IN THE DATA SET, AS A RESULT I CHANGED THE CV TO STRIFIEDKFOLD N_SPLITS = 5, AND ADDED A PARAMETER CLASS_WEIGHT = 'BALANCED' ADDRESS THIS ISSUE. the performance metrics went from 0.93 to 0.99 and accuracy from 0.93 to 0.98

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold  
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

# Uses GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Data Loading & Preparation
df = pd.read_excel('../minmax.xlsx')
data = df.values
labels = pd.read_csv('../idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Autoencoder-Classifier Model
class JointAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointAutoencoderClassifier, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh()  # Ensure your input is in [-1, 1]. Use Sigmoid() if [0, 1]
        )
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        logits = self.classifier(latent)
        return reconstruction, logits

# 3. Loss Functions & Optimizer
reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
lambda_recon = 0.5

def combined_loss(reconstructed, original, logits, labels):
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    loss_class = classification_loss_fn(logits, labels)
    return lambda_recon * loss_recon + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32

model = JointAutoencoderClassifier(input_dim, latent_dim, num_classes).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50

# 4. Joint Training
print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        reconstruction, logits = model(inputs)
        loss = combined_loss(reconstruction, inputs, logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# 5. Extract Latent Features
model.eval()
with torch.no_grad():
    train_latent = model.encoder(X_train.to(device)).cpu().numpy()
    test_latent = model.encoder(X_test.to(device)).cpu().numpy()

y_train_np = y_train.numpy()
y_test_np = y_test.numpy()



# 6. Modified BGWO Feature Selection
def bgwo_optimize(features, labels, num_wolves=10, max_iterations=20):
    num_features = features.shape[1]
    
    # Initialize wolves with at least one feature
    wolves = np.random.randint(0, 2, size=(num_wolves, num_features))
    for wolf in wolves:
        if not wolf.any():
            wolf[np.random.randint(0, num_features)] = 1

    alpha, beta, delta = wolves[0], wolves[1], wolves[2]
    alpha_fitness, beta_fitness, delta_fitness = -np.inf, -np.inf, -np.inf

    def fitness_function(position):
        selected = np.where(position == 1)[0]
        if len(selected) == 0:
            return -1  # Penalize empty selections
        try:
            
            model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
            scores = cross_val_score(model, features[:, selected], labels, StratifiedKFold(n_splits=5), scoring='accuracy')


            return np.mean(scores)
        except:
            return -1

    for iteration in range(max_iterations):
        # Evaluate fitness for all wolves
        fitness = np.array([fitness_function(wolf) for wolf in wolves])
        
        # Update alpha, beta, delta
        for i in range(num_wolves):
            if fitness[i] > alpha_fitness:
                delta, delta_fitness = beta, beta_fitness
                beta, beta_fitness = alpha, alpha_fitness
                alpha, alpha_fitness = wolves[i].copy(), fitness[i]
            elif fitness[i] > beta_fitness:
                delta, delta_fitness = beta, beta_fitness
                beta, beta_fitness = wolves[i].copy(), fitness[i]
            elif fitness[i] > delta_fitness:
                delta, delta_fitness = wolves[i].copy(), fitness[i]

        # Update wolves' positions
        a = 2 - 2 * iteration / max_iterations  # Decreases linearly from 2 to 0
        
        for i in range(num_wolves):
            for j in range(num_features):
                # Calculate distances and new positions
                X_alpha = alpha[j] - a * (2 * np.random.rand() - 1) * abs(alpha[j] - wolves[i,j])
                X_beta = beta[j] - a * (2 * np.random.rand() - 1) * abs(beta[j] - wolves[i,j])
                X_delta = delta[j] - a * (2 * np.random.rand() - 1) * abs(delta[j] - wolves[i,j])
                
                # Update position
                new_pos = (X_alpha + X_beta + X_delta) / 3
                wolves[i,j] = 1 if new_pos > 0.5 else 0

        print(f"Iteration {iteration+1}: Best Fitness = {alpha_fitness:.4f}")
        print(f"Selected Features: {np.where(alpha == 1)[0]}")

    return np.where(alpha == 1)[0]

selected_features = bgwo_optimize(train_latent, y_train_np)

# 7. Train & Evaluate SVM
if len(selected_features) > 0:
    print(f"\nFinal selected features: {selected_features}")
    
    svm_model = SVC(random_state=42, class_weight='balanced')
    svm_model.fit(train_latent[:, selected_features], y_train_np)
    svm_preds = svm_model.predict(test_latent[:, selected_features])
    svm_acc = accuracy_score(y_test_np, svm_preds)
    print('\nSVM Performance with BGWO-selected features:')
    print('Confusion Matrix:\n', confusion_matrix(y_test_np, svm_preds))
    print('\nClassification Report:\n', classification_report(y_test_np, svm_preds))
    print(f"\nSVM Accuracy: {svm_acc * 100:.2f}%")
    
    # Random Forest Classifier
    
    # rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    # rf_model.fit(train_latent[:, selected_features], y_train_np)
    # rf_preds = rf_model.predict(test_latent[:, selected_features])
    # rf_acc = accuracy_score(y_test_np, rf_preds)
    
    # print('\nRandom Forest Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, rf_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, rf_preds))
    # print(f"\nRF Accuracy: {rf_acc * 100:.2f}%")
    
    
    # Naive Bayes Classifier
    
    # nb_model = GaussianNB()
    # nb_model.fit(train_latent[:, selected_features], y_train_np)
    # nb_preds = nb_model.predict(test_latent[:, selected_features])
    # nb_acc = accuracy_score(y_test_np, nb_preds)
    
    # print('\nNaive Bayes Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, nb_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, nb_preds))
    # print(f"\nNBC Accuracy: {nb_acc * 100:.2f}%")
    
    
     
else:
    # Fallback: Use all features if BGWO fails
    print("\nWarning: BGWO selected no features. Using all features as fallback.")
    
    # SVM Classifier
    
    svm_model = SVC(random_state=42, class_weight='balanced')
    svm_model.fit(train_latent, y_train_np)
    svm_preds = svm_model.predict(test_latent)
    svm_acc = accuracy_score(y_test_np, svm_preds)
    
    print('\nSVM Performance (using all features):')
    print('Confusion Matrix:\n', confusion_matrix(y_test_np, svm_preds))
    print('\nClassification Report:\n', classification_report(y_test_np, svm_preds))
    print(f"\nSVM Accuracy: {svm_acc * 100:.2f}%")
    
    
    # Random Forest Classifier
    # rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    # rf_model.fit(train_latent[:, selected_features], y_train_np)
    # rf_preds = rf_model.predict(test_latent[:, selected_features])
    # rf_acc = accuracy_score(y_test_np, rf_preds)
    
    # print('\nRandom Forest Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, rf_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, rf_preds))
    # print(f"\nRF Accuracy: {rf_acc * 100:.2f}%")
    
    # Naive Bayes Classifier 
    # nb_model = GaussianNB()
    # nb_model.fit(train_latent, y_train_np)
    # nb_preds = nb_model.predict(test_latent)
    # nb_acc = accuracy_score(y_test_np, nb_preds)
    
    # print('\nNaive Bayes Performance (using all features):')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, nb_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, nb_preds))
    # print(f"\nNBC Accuracy: {nb_acc * 100:.2f}%")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.2080
Epoch [2/50], Loss: 0.8043
Epoch [3/50], Loss: 0.6275
Epoch [4/50], Loss: 0.5959
Epoch [5/50], Loss: 0.4858
Epoch [6/50], Loss: 0.4860
Epoch [7/50], Loss: 0.4087
Epoch [8/50], Loss: 0.4133
Epoch [9/50], Loss: 0.4791
Epoch [10/50], Loss: 0.4834
Epoch [11/50], Loss: 0.4189
Epoch [12/50], Loss: 0.4494
Epoch [13/50], Loss: 0.3856
Epoch [14/50], Loss: 0.4638
Epoch [15/50], Loss: 0.4051
Epoch [16/50], Loss: 0.4045
Epoch [17/50], Loss: 0.4420
Epoch [18/50], Loss: 0.4197
Epoch [19/50], Loss: 0.4011
Epoch [20/50], Loss: 0.3821
Epoch [21/50], Loss: 0.3867
Epoch [22/50], Loss: 0.3603
Epoch [23/50], Loss: 0.4178
Epoch [24/50], Loss: 0.3771
Epoch [25/50], Loss: 0.3556
Epoch [26/50], Loss: 0.3831
Epoch [27/50], Loss: 0.4171
Epoch [28/50], Loss: 0.3888
Epoch [29/50], Loss: 0.3987
Epoch [30/50], Loss: 0.3701
Epoch [31/50], Loss: 0.4412
Epoch [32/50], Loss: 0.3827
Epoch [33/50], Loss: 0.3966
Epoch [34/50], Loss: 0.3976
Epoch [35/50], Loss: 0.4004


SVM + AE + BGWO for feature selection 

IMPORTANT NOTE:

**USING CV = 5**
**CLASS_WEIGHT = 'BALANCED'**

FOR EXAMPLE OUTPUTS WARNINGS BECAUSE A CLASS HAS 2 MEMBERS ONLY IN THE DATA SET, AS A RESULT I CHANGED THE CV TO STRIFIEDKFOLD N_SPLITS = 5, AND ADDED A PARAMETER CLASS_WEIGHT = 'BALANCED' ADDRESS THIS ISSUE. the performance metrics went from 0.93 to 0.99 and accuracy from 0.93 to 0.98

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold  
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

# Uses GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Data Loading & Preparation
df = pd.read_excel('../minmax.xlsx')
data = df.values
labels = pd.read_csv('../idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Autoencoder-Classifier Model
class JointAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointAutoencoderClassifier, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh()  # Ensure your input is in [-1, 1]. Use Sigmoid() if [0, 1]
        )
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        logits = self.classifier(latent)
        return reconstruction, logits

# 3. Loss Functions & Optimizer
reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
lambda_recon = 0.5

def combined_loss(reconstructed, original, logits, labels):
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    loss_class = classification_loss_fn(logits, labels)
    return lambda_recon * loss_recon + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32

model = JointAutoencoderClassifier(input_dim, latent_dim, num_classes).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50

# 4. Joint Training
print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        reconstruction, logits = model(inputs)
        loss = combined_loss(reconstruction, inputs, logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# 5. Extract Latent Features
model.eval()
with torch.no_grad():
    train_latent = model.encoder(X_train.to(device)).cpu().numpy()
    test_latent = model.encoder(X_test.to(device)).cpu().numpy()

y_train_np = y_train.numpy()
y_test_np = y_test.numpy()



# 6. Modified BGWO Feature Selection
def bgwo_optimize(features, labels, num_wolves=10, max_iterations=20):
    num_features = features.shape[1]
    
    # Initialize wolves with at least one feature
    wolves = np.random.randint(0, 2, size=(num_wolves, num_features))
    for wolf in wolves:
        if not wolf.any():
            wolf[np.random.randint(0, num_features)] = 1

    alpha, beta, delta = wolves[0], wolves[1], wolves[2]
    alpha_fitness, beta_fitness, delta_fitness = -np.inf, -np.inf, -np.inf

    def fitness_function(position):
        selected = np.where(position == 1)[0]
        if len(selected) == 0:
            return -1  # Penalize empty selections
        try:
            
            model = SVC(random_state=42,class_weight='balanced')
            scores = cross_val_score(model, features[:, selected], labels, StratifiedKFold(n_splits=5), scoring='accuracy')
            return np.mean(scores)
        except:
            return -1

    for iteration in range(max_iterations):
        # Evaluate fitness for all wolves
        fitness = np.array([fitness_function(wolf) for wolf in wolves])
        
        # Update alpha, beta, delta
        for i in range(num_wolves):
            if fitness[i] > alpha_fitness:
                delta, delta_fitness = beta, beta_fitness
                beta, beta_fitness = alpha, alpha_fitness
                alpha, alpha_fitness = wolves[i].copy(), fitness[i]
            elif fitness[i] > beta_fitness:
                delta, delta_fitness = beta, beta_fitness
                beta, beta_fitness = wolves[i].copy(), fitness[i]
            elif fitness[i] > delta_fitness:
                delta, delta_fitness = wolves[i].copy(), fitness[i]

        # Update wolves' positions
        a = 2 - 2 * iteration / max_iterations  # Decreases linearly from 2 to 0
        
        for i in range(num_wolves):
            for j in range(num_features):
                # Calculate distances and new positions
                X_alpha = alpha[j] - a * (2 * np.random.rand() - 1) * abs(alpha[j] - wolves[i,j])
                X_beta = beta[j] - a * (2 * np.random.rand() - 1) * abs(beta[j] - wolves[i,j])
                X_delta = delta[j] - a * (2 * np.random.rand() - 1) * abs(delta[j] - wolves[i,j])
                
                # Update position
                new_pos = (X_alpha + X_beta + X_delta) / 3
                wolves[i,j] = 1 if new_pos > 0.5 else 0

        print(f"Iteration {iteration+1}: Best Fitness = {alpha_fitness:.4f}")
        print(f"Selected Features: {np.where(alpha == 1)[0]}")

    return np.where(alpha == 1)[0]

selected_features = bgwo_optimize(train_latent, y_train_np)

# 7. Train & Evaluate SVM
if len(selected_features) > 0:
    print(f"\nFinal selected features: {selected_features}")
    svm_model = SVC(random_state=42, class_weight='balanced')
    svm_model.fit(train_latent[:, selected_features], y_train_np)
    svm_preds = svm_model.predict(test_latent[:, selected_features])
    svm_acc = accuracy_score(y_test_np, svm_preds)
    print('\nSVM Performance with BGWO-selected features:')
    print('Confusion Matrix:\n', confusion_matrix(y_test_np, svm_preds))
    print('\nClassification Report:\n', classification_report(y_test_np, svm_preds))
    print(f"\nSVM Accuracy: {svm_acc * 100:.2f}%")
    
    # Random Forest Classifier
    
    # rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    # rf_model.fit(train_latent[:, selected_features], y_train_np)
    # rf_preds = rf_model.predict(test_latent[:, selected_features])
    # rf_acc = accuracy_score(y_test_np, rf_preds)
    
    # print('\nRandom Forest Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, rf_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, rf_preds))
    # print(f"\nRF Accuracy: {rf_acc * 100:.2f}%")
    
    
    # Naive Bayes Classifier
    
    # nb_model = GaussianNB()
    # nb_model.fit(train_latent[:, selected_features], y_train_np)
    # nb_preds = nb_model.predict(test_latent[:, selected_features])
    # nb_acc = accuracy_score(y_test_np, nb_preds)
    
    # print('\nNaive Bayes Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, nb_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, nb_preds))
    # print(f"\nNBC Accuracy: {nb_acc * 100:.2f}%")
    
    
     
else:
    # Fallback: Use all features if BGWO fails
    print("\nWarning: BGWO selected no features. Using all features as fallback.")
    svm_model = SVC(random_state=42, class_weight='balanced')
    svm_model.fit(train_latent, y_train_np)
    svm_preds = svm_model.predict(test_latent)
    svm_acc = accuracy_score(y_test_np, svm_preds)
    
    print('\nSVM Performance (using all features):')
    print('Confusion Matrix:\n', confusion_matrix(y_test_np, svm_preds))
    print('\nClassification Report:\n', classification_report(y_test_np, svm_preds))
    print(f"\nSVM Accuracy: {svm_acc * 100:.2f}%")
    
    
    # rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    # rf_model.fit(train_latent[:, selected_features], y_train_np)
    # rf_preds = rf_model.predict(test_latent[:, selected_features])
    # rf_acc = accuracy_score(y_test_np, rf_preds)
    
    # print('\nRandom Forest Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, rf_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, rf_preds))
    # print(f"\nRF Accuracy: {rf_acc * 100:.2f}%")
    
    # Naive Bayes Classifier 
    # nb_model = GaussianNB()
    # nb_model.fit(train_latent, y_train_np)
    # nb_preds = nb_model.predict(test_latent)
    # nb_acc = accuracy_score(y_test_np, nb_preds)
    
    # print('\nNaive Bayes Performance (using all features):')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, nb_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, nb_preds))
    # print(f"\nNBC Accuracy: {nb_acc * 100:.2f}%")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.1992
Epoch [2/50], Loss: 0.8383
Epoch [3/50], Loss: 0.6952
Epoch [4/50], Loss: 0.5506
Epoch [5/50], Loss: 0.5085
Epoch [6/50], Loss: 0.4624
Epoch [7/50], Loss: 0.4693
Epoch [8/50], Loss: 0.4632
Epoch [9/50], Loss: 0.4446
Epoch [10/50], Loss: 0.4570
Epoch [11/50], Loss: 0.4135
Epoch [12/50], Loss: 0.4052
Epoch [13/50], Loss: 0.4076
Epoch [14/50], Loss: 0.4156
Epoch [15/50], Loss: 0.4137
Epoch [16/50], Loss: 0.3928
Epoch [17/50], Loss: 0.3963
Epoch [18/50], Loss: 0.3917
Epoch [19/50], Loss: 0.4089
Epoch [20/50], Loss: 0.4034
Epoch [21/50], Loss: 0.3558
Epoch [22/50], Loss: 0.4014
Epoch [23/50], Loss: 0.3578
Epoch [24/50], Loss: 0.3666
Epoch [25/50], Loss: 0.4056
Epoch [26/50], Loss: 0.4011
Epoch [27/50], Loss: 0.3799
Epoch [28/50], Loss: 0.3930
Epoch [29/50], Loss: 0.4022
Epoch [30/50], Loss: 0.3748
Epoch [31/50], Loss: 0.3898
Epoch [32/50], Loss: 0.3598
Epoch [33/50], Loss: 0.3738
Epoch [34/50], Loss: 0.3846
Epoch [35/50], Loss: 0.4163


RF + AE + BGWO for feature selection 

IMPORTANT NOTE:

**USING CV = 5**
**CLASS_WEIGHT = 'BALANCED'**

FOR EXAMPLE OUTPUTS WARNINGS BECAUSE A CLASS HAS 2 MEMBERS ONLY IN THE DATA SET, AS A RESULT I CHANGED THE CV TO STRIFIEDKFOLD N_SPLITS = 5, AND ADDED A PARAMETER CLASS_WEIGHT = 'BALANCED' ADDRESS THIS ISSUE. the performance metrics went from 0.93 to 0.99 and accuracy from 0.93 to 0.98

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold  
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

# Uses GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Data Loading & Preparation
df = pd.read_excel('../minmax.xlsx')
data = df.values
labels = pd.read_csv('../idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Autoencoder-Classifier Model
class JointAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointAutoencoderClassifier, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh()  # Ensure your input is in [-1, 1]. Use Sigmoid() if [0, 1]
        )
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        logits = self.classifier(latent)
        return reconstruction, logits

# 3. Loss Functions & Optimizer
reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
lambda_recon = 0.5

def combined_loss(reconstructed, original, logits, labels):
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    loss_class = classification_loss_fn(logits, labels)
    return lambda_recon * loss_recon + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32

model = JointAutoencoderClassifier(input_dim, latent_dim, num_classes).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50

# 4. Joint Training
print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        reconstruction, logits = model(inputs)
        loss = combined_loss(reconstruction, inputs, logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# 5. Extract Latent Features
model.eval()
with torch.no_grad():
    train_latent = model.encoder(X_train.to(device)).cpu().numpy()
    test_latent = model.encoder(X_test.to(device)).cpu().numpy()

y_train_np = y_train.numpy()
y_test_np = y_test.numpy()



# 6. Modified BGWO Feature Selection
def bgwo_optimize(features, labels, num_wolves=10, max_iterations=20):
    num_features = features.shape[1]
    
    # Initialize wolves with at least one feature
    wolves = np.random.randint(0, 2, size=(num_wolves, num_features))
    for wolf in wolves:
        if not wolf.any():
            wolf[np.random.randint(0, num_features)] = 1

    alpha, beta, delta = wolves[0], wolves[1], wolves[2]
    alpha_fitness, beta_fitness, delta_fitness = -np.inf, -np.inf, -np.inf

    def fitness_function(position):
        selected = np.where(position == 1)[0]
        if len(selected) == 0:
            return -1  # Penalize empty selections
        try:
            
            model = GaussianNB()
            scores = cross_val_score(model, features[:, selected], labels, StratifiedKFold(n_splits=5), scoring='accuracy')


            return np.mean(scores)
        except:
            return -1

    for iteration in range(max_iterations):
        # Evaluate fitness for all wolves
        fitness = np.array([fitness_function(wolf) for wolf in wolves])
        
        # Update alpha, beta, delta
        for i in range(num_wolves):
            if fitness[i] > alpha_fitness:
                delta, delta_fitness = beta, beta_fitness
                beta, beta_fitness = alpha, alpha_fitness
                alpha, alpha_fitness = wolves[i].copy(), fitness[i]
            elif fitness[i] > beta_fitness:
                delta, delta_fitness = beta, beta_fitness
                beta, beta_fitness = wolves[i].copy(), fitness[i]
            elif fitness[i] > delta_fitness:
                delta, delta_fitness = wolves[i].copy(), fitness[i]

        # Update wolves' positions
        a = 2 - 2 * iteration / max_iterations  # Decreases linearly from 2 to 0
        
        for i in range(num_wolves):
            for j in range(num_features):
                # Calculate distances and new positions
                X_alpha = alpha[j] - a * (2 * np.random.rand() - 1) * abs(alpha[j] - wolves[i,j])
                X_beta = beta[j] - a * (2 * np.random.rand() - 1) * abs(beta[j] - wolves[i,j])
                X_delta = delta[j] - a * (2 * np.random.rand() - 1) * abs(delta[j] - wolves[i,j])
                
                # Update position
                new_pos = (X_alpha + X_beta + X_delta) / 3
                wolves[i,j] = 1 if new_pos > 0.5 else 0

        print(f"Iteration {iteration+1}: Best Fitness = {alpha_fitness:.4f}")
        print(f"Selected Features: {np.where(alpha == 1)[0]}")

    return np.where(alpha == 1)[0]

selected_features = bgwo_optimize(train_latent, y_train_np)

# 7. Train & Evaluate SVM
if len(selected_features) > 0:
    print(f"\nFinal selected features: {selected_features}")
    
    # svm_model = SVC(random_state=42, class_weight='balanced')
    # svm_model.fit(train_latent[:, selected_features], y_train_np)
    # svm_preds = svm_model.predict(test_latent[:, selected_features])
    # svm_acc = accuracy_score(y_test_np, svm_preds)
    # print('\nSVM Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, svm_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, svm_preds))
    # print(f"\nSVM Accuracy: {svm_acc * 100:.2f}%")
    
    # Random Forest Classifier
    
    # rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    # rf_model.fit(train_latent[:, selected_features], y_train_np)
    # rf_preds = rf_model.predict(test_latent[:, selected_features])
    # rf_acc = accuracy_score(y_test_np, rf_preds)
    
    # print('\nRandom Forest Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, rf_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, rf_preds))
    # print(f"\nRF Accuracy: {rf_acc * 100:.2f}%")
    
    
    # Naive Bayes Classifier
    
    nb_model = GaussianNB()
    nb_model.fit(train_latent[:, selected_features], y_train_np)
    nb_preds = nb_model.predict(test_latent[:, selected_features])
    nb_acc = accuracy_score(y_test_np, nb_preds)
    
    print('\nNaive Bayes Performance with BGWO-selected features:')
    print('Confusion Matrix:\n', confusion_matrix(y_test_np, nb_preds))
    print('\nClassification Report:\n', classification_report(y_test_np, nb_preds))
    print(f"\nNBC Accuracy: {nb_acc * 100:.2f}%")
    
    
     
else:
    # Fallback: Use all features if BGWO fails
    print("\nWarning: BGWO selected no features. Using all features as fallback.")
    
    # SVM Classifier
    
    # svm_model = SVC(random_state=42, class_weight='balanced')
    # svm_model.fit(train_latent, y_train_np)
    # svm_preds = svm_model.predict(test_latent)
    # svm_acc = accuracy_score(y_test_np, svm_preds)
    
    # print('\nSVM Performance (using all features):')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, svm_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, svm_preds))
    # print(f"\nSVM Accuracy: {svm_acc * 100:.2f}%")
    
    
    # Random Forest Classifier
    # rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    # rf_model.fit(train_latent[:, selected_features], y_train_np)
    # rf_preds = rf_model.predict(test_latent[:, selected_features])
    # rf_acc = accuracy_score(y_test_np, rf_preds)
    
    # print('\nRandom Forest Performance with BGWO-selected features:')
    # print('Confusion Matrix:\n', confusion_matrix(y_test_np, rf_preds))
    # print('\nClassification Report:\n', classification_report(y_test_np, rf_preds))
    # print(f"\nRF Accuracy: {rf_acc * 100:.2f}%")
    
    # Naive Bayes Classifier 
    nb_model = GaussianNB()
    nb_model.fit(train_latent, y_train_np)
    nb_preds = nb_model.predict(test_latent)
    nb_acc = accuracy_score(y_test_np, nb_preds)
    
    print('\nNaive Bayes Performance (using all features):')
    print('Confusion Matrix:\n', confusion_matrix(y_test_np, nb_preds))
    print('\nClassification Report:\n', classification_report(y_test_np, nb_preds))
    print(f"\nNBC Accuracy: {nb_acc * 100:.2f}%")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.1850
Epoch [2/50], Loss: 0.8321
Epoch [3/50], Loss: 0.6479
Epoch [4/50], Loss: 0.5666
Epoch [5/50], Loss: 0.4960
Epoch [6/50], Loss: 0.4806
Epoch [7/50], Loss: 0.4914
Epoch [8/50], Loss: 0.4845
Epoch [9/50], Loss: 0.4505
Epoch [10/50], Loss: 0.4819
Epoch [11/50], Loss: 0.4338
Epoch [12/50], Loss: 0.4131
Epoch [13/50], Loss: 0.3946
Epoch [14/50], Loss: 0.3818
Epoch [15/50], Loss: 0.4275
Epoch [16/50], Loss: 0.4113
Epoch [17/50], Loss: 0.4676
Epoch [18/50], Loss: 0.4457
Epoch [19/50], Loss: 0.4561
Epoch [20/50], Loss: 0.4162
Epoch [21/50], Loss: 0.4553
Epoch [22/50], Loss: 0.3944
Epoch [23/50], Loss: 0.4436
Epoch [24/50], Loss: 0.3766
Epoch [25/50], Loss: 0.4417
Epoch [26/50], Loss: 0.3862
Epoch [27/50], Loss: 0.3664
Epoch [28/50], Loss: 0.3670
Epoch [29/50], Loss: 0.3926
Epoch [30/50], Loss: 0.3522
Epoch [31/50], Loss: 0.3950
Epoch [32/50], Loss: 0.3842
Epoch [33/50], Loss: 0.3469
Epoch [34/50], Loss: 0.3435
Epoch [35/50], Loss: 0.3605
